# C1.10 · Forensic replay and control architecture

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.9 · Machine-speed containment and fleet revocation](https://spbreed.github.io/cyber-commons/lessons/C1.9.html)**.

| | |
|---|---|
| Tools used | Velociraptor |

## What this lesson is

**What it covers.** Locking the deterministic runtime constants — prompts, tool results, model version, seed — so a complex agentic exploit path reproduces inside a forensic lab.

**Why a security engineer needs it.** A finding is only as good as its reproduction, and the model version is the constant teams miss and the one that silently invalidates the rest. A run you cannot reproduce is a story.

## 1 · The hook

A finding is only reproducible if the run is. Miss one of the four constants — usually the model version — and you can describe the exploit but never demonstrate it, which is the moment your conclusion stops being defensible.

> **At CyberTravels.** The run is the CyberTravels refund incident, and the field missing most often is the advisor's model version — upgraded by the provider between the incident and the replay.

## 2 · The framework

```
   reproduce a run from four constants

   prompts + tool results + model version + seed  -> identical run
                              ^
                              |
              miss this one and every other constant describes
              a system the provider has since replaced

   a run you cannot reproduce is a story, not evidence.
```

A finding is only reproducible if the run is. Forensic replay locks the
**deterministic runtime constraints** — the prompts, the tool results, the
pinned model version and the sampling seed — so a complex agentic exploit path
can be reproduced, replayed and documented inside an isolated forensic lab
rather than described from memory.

The field teams miss most often is the model version, and it is the one that
silently invalidates everything else: a provider-side upgrade changes the
behaviour under every other constant, so a replay that does not pin it is
reproducing a different system.

## 3 · The procedure, as a skill

The skill audits a CyberTravels run for the four fields a replay needs, and reports which are missing — because a run you cannot reproduce is a story, not evidence.

### The skill — [`skills/response/run-replayability-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/run-replayability-audit/SKILL.md)

```yaml
name: run-replayability-audit
description: >-
  Check whether an incident run can be replayed at all — model version, seed,
  prompt, tool results, retrieved context — and what a later model version does
  to the replay. Use when forensics needs to know why the agent did what it did.
allowed-tools: Read, Grep, Glob
```

# Replay needs five things and production records three

"Why did the agent do that" is answerable only if the run can be re-run under the
conditions it ran in. That needs the model version, the sampling parameters and
seed, the exact prompt, every tool result and the retrieved context. A typical
production run records enough to see what happened and not enough to reproduce
it.

## When to use this

Before an incident, as a readiness check, and during one, to establish honestly
whether the reconstruction is possible.

## Procedure

**1 — List the five inputs and check each against a real run record.** Model
version and seed are the two usually missing, and their absence is decisive
rather than inconvenient.

**2 — Attempt the replay.** If any input is missing, say what the replay can and
cannot establish. A partial replay is still useful for the tool path and useless
for the reasoning.

**3 — Replay under later model versions.** The provider has probably upgraded.
Record whether the action changes: if it does, the original decision cannot be
reproduced at all, and that is a finding about the estate rather than about the
incident.

**4 — Cost full instrumentation.** Storage and latency for recording everything,
against the incidents where you needed it. Present both; the answer is usually to
instrument the high-tier agents only, and that is a defensible decision when the
numbers are attached.

**5 — Record what the estate has chosen.** Which agents are replayable and which
are not, so nobody assumes during an incident.

## Example

**Input** — the fixture committed at the top of [`scripts/run_replayability_audit.py`](scripts/run_replayability_audit.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
fully instrumented     replayable=True
typical production     replayable=False
      ✗ model version — a silent upgrade changes the output
      ✗ seed — sampling makes the run unrepeatable
prompts only           replayable=False
      ✗ tool results — the agent saw a world you cannot rebuild
      ✗ model version — a silent upgrade changes the output
      ✗ seed — sampling makes the run unrepeatable
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "inputs": [{"name": "str", "recorded": false}],
  "replay": {"possible": false, "establishes": ["str"], "cannot_establish": ["str"]},
  "version_drift": [{"version": "str", "action": "str", "same_as_original": false}],
  "cost": {"storage_per_run": "str", "latency_ms": 0},
  "policy": [{"tier": "str", "fully_instrumented": true}]
}
```

## Failure modes

- **Assuming replay is possible.** Check the record before promising it.
- **Replaying on the current model.** It is not the one that acted.
- **Instrumenting everything or nothing.** Tier it and write the choice down.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/run-replayability-audit/scripts/run_replayability_audit.py
SCRIPT = "skills/response/run-replayability-audit/scripts/run_replayability_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

A run with all four fields replays identically; one missing the model version cannot be demonstrated, only described, which is the moment the finding stops being defensible.

## Your turn

Pick one agent run from last week and try to reproduce it. The first field you cannot recover is the one to start logging today.

---

**Next → [C1.11 · Institutional governance and compliance](https://spbreed.github.io/cyber-commons/lessons/C1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*